# Chapter 22: Agent-Based Modeling with DisSModel

*Part IV — DisSModel: Core and Paradigms*

Implemented by the [`dissmodel-abm`](https://github.com/DisSModel/dissmodel-abm) package.

## Learning Objectives

By the end of this chapter you will be able to:

- Understand the `Society`/`Agent` protective layer over the vector substrate
- Translate `Agent`/`Society` concepts from TerraME to `dissmodel-abm`
- Write an agent-based model without touching `self.gdf` directly
- Know which models ship today, and what's explicitly still missing

In [1]:
# Standard imports
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

Chapter 21's cellular automata all shared one constraint: a fixed grid, one rule applied identically to every cell, cells that neither move nor disappear. Real agents don't cooperate with that constraint — they walk, they compete for space, they're born and they die mid-run. This chapter is where DisSModel stops pretending every spatial actor is a stationary cell.

## Society and Agent: A Protective Layer

The whole point of `dissmodel-abm` is that model code never touches `self.gdf` directly. Killing an agent whose energy has run out, in raw `GeoDataFrame` terms, looks like this:

```python
self.gdf = self.gdf[self.gdf["energy"] > 0].reset_index(drop=True)
```

Through `self.society`, the same rule reads as an object-oriented loop:

```python
for agent in self.society:
    if agent.energy <= 0:
        agent.die()
```

`Society` owns no data of its own — it reads and writes through the host model's `gdf` attribute, so `model.gdf` and `model.society` are always views onto the same rows. `Agent` is a thin proxy over one row: `agent.energy = 5` writes the underlying cell directly, no separate copy to keep synchronized. `Map`, `Chart`, and every `ModelExecutor` from Chapter 19 keep working completely unmodified underneath, whether or not a given model's `execute()` ever mentions `self.society` — `AgentModel` is a `SpatialModel` subclass with a lazily-created `society` property, not a parallel class hierarchy competing with Chapter 18's lifecycle.

In [2]:
import geopandas as gpd
import numpy as np
from dissmodel.core import Environment, Model
from dissmodel_abm.core import AgentModel

n = 20
bounds = (0, 0, 100, 100)
rng = np.random.default_rng(42)
gdf = gpd.GeoDataFrame({
    "energy": rng.uniform(5, 12, n),
    "geometry": gpd.points_from_xy(
        rng.uniform(bounds[0], bounds[2], n),
        rng.uniform(bounds[1], bounds[3], n),
    ),
})

class EnergyDrain(AgentModel):
    def execute(self):
        for agent in self.society:
            agent.energy -= 2.0
            if agent.energy <= 0:
                agent.die()

env = Environment(start_time=0, end_time=3)
model = EnergyDrain(gdf=gdf)
print("Agents before:", len(model.gdf))
env.run()
print("Agents after: ", len(model.gdf))

Agents before: 20
Running from 0 to 3 (duration: 3)
Agents after:  15


**Agents without a location.** Following TerraME — where an agent may exist with no placement until it's explicitly given one — an agent here can exist with `geometry = None`: `society.add(energy=4.0)` creates one, `agent.has_location` reports `False`, and `agent.enter(x, y)` gives it a position later. Calling a spatial method (`walk`, `neighbors`, `distance_to`) on a location-less agent raises a clear `RuntimeError` rather than failing deep inside `geopandas` on a `None` geometry.

One structural difference is worth naming explicitly: TerraME agents are autonomous objects that carry their own behavior, each with its own `execute`. `dissmodel-abm` agents are data with a uniform interface — behavior lives once, in the owning model's `execute()`, applied identically to every agent via `for agent in self.society`. That trades TerraME's per-agent heterogeneous behavior for staying close to a vectorizable substrate, the identical trade-off Chapter 21 already made for `CellularAutomaton.rule(idx)`.

## Concept Mapping: TerraME to dissmodel-abm

| TerraME (`Agent`/`Society`) | `dissmodel-abm` |
|---|---|
| `execute(self)` | `execute()` (`Model` lifecycle, Chapter 18) |
| `init(self)` | `setup()` (`Model` lifecycle, Chapter 18) |
| `Society` (collection of Agents) | `self.society` — object-oriented view over `self.gdf` |
| `Agent` | `self.society[idx]` — a proxy over one row |
| `placement` / `getCell()` | `agent.geometry` |
| `enter(cell)` | `agent.enter(x, y)` |
| `leave()` | `agent.leave()` |
| `move(cell)` / `walk()` | `agent.move_to(x, y)` / `agent.walk(step_size, bounds)` |
| `die()` | `agent.die()` |
| `reproduce()` | `agent.reproduce(**overrides)` |
| neighborhood | `agent.neighbors(radius)` (points) or `agent.grid_neighbors()` (cells) |
| `Society:add` / `Society:remove` | `society.add(**attrs)` / `society.remove(agent_or_idx)` |
| `forEachAgent` | `for agent in society: ...` |
| `addSocialNetwork` / `message` | not provided yet |
| `State` / `Jump` / `Flow` | not provided yet |

Two agent layouts recur across the shipped models. **Point-agent** models (`RandomWalkModel`, `PredatorPreyModel`) back `self.society` with `Point` geometry plus ordinary state columns — agents move continuously through space. **One-agent-per-cell** models (`SchellingModel`) back it with a polygon grid instead, `vector_grid()` from Chapter 7 — agents occupy discrete cells and can only move to another empty one.

## Installation and Quick Start

Like every other extension package since Chapter 19, `dissmodel-abm` has no PyPI release yet:

```bash
git clone https://github.com/DisSModel/dissmodel-abm.git
cd dissmodel-abm
pip install -e .
```

Writing a new model looks exactly like writing a `dissmodel-ca` model (Chapter 21), swapping `CellularAutomaton.rule(idx)` for a `society` loop:

```python
from dissmodel_abm.core import AgentModel

class MyModel(AgentModel):
    def setup(self, **params):
        ...  # one-time initialization

    def execute(self):
        for agent in self.society:
            agent.walk(step_size=1.0, bounds=(0, 0, 100, 100))
            if agent.energy <= 0:
                agent.die()
            elif agent.energy >= 15:
                agent.reproduce(energy=5.0)
```

The shipped `RandomWalkModel` is the minimal working version of exactly that pattern:

In [3]:
from dissmodel_abm.models import RandomWalkModel

env = Environment(start_time=0, end_time=20)
walk_model = RandomWalkModel(gdf=gdf.copy(), step_size=2.0, bounds=bounds)
env.run()
print("Sample agent final position:", walk_model.gdf.geometry.iloc[0])

Running from 0 to 20 (duration: 20)
Sample agent final position: POINT (68.89006559875239 48.232925176711504)


## Theory: Bottom-Up Modeling

Agent-based modeling appears under several names across the literature — ABM, multi-agent systems, individual-based modeling — spanning economics, sociology, ecology, and political science. What unifies them is a **bottom-up** approach: complex system behavior emerges from the interaction of discrete agents, rather than being specified as an aggregate equation the way Chapter 20's system dynamics models are. An **agent** is any actor able to affect itself, its environment, and other agents.

Helen Couclelis's classification of ABM applications, along two axes — natural versus artificial agent, natural versus artificial environment — places most of the models in this book in the same quadrant:

| | Natural environment | Artificial environment |
|---|---|---|
| **Natural agent** | Behavioral experiments | Descriptive model |
| **Artificial agent** | Engineering applications | e-science |

`PredatorPreyModel`, coming up next, sits squarely in "descriptive model" — artificial agents standing in for real animals, inside a deliberately simplified artificial environment.

Nigel Gilbert's case for why ABM is worth its extra complexity, relative to Chapter 20's aggregate models, comes down to three things a bottom-up model represents directly instead of assuming: **structure** (it emerges from agent interaction rather than being imposed from outside), **agency** (agents have goals and beliefs that drive their actions), and **dynamics** (agents move, learn, and change position — spatially and socially — over the course of a run). ABM also handles qualitative and relational data System Dynamics' continuous aggregate quantities simply can't represent — who is a given type, who is adjacent to whom.

## Case Study: Predator-Prey, From Equation to Individual

Chapter 20 modeled predator and prey as two continuous stocks under Lotka-Volterra. This section rebuilds the same phenomenon bottom-up, translating each ODE parameter into an individual agent rule:

| ODE parameter | Agent rule |
|---|---|
| `r` — prey growth | eating pasture raises energy; above a threshold, reproduce (energy halved) |
| `m` — predator mortality | dies at energy ≤ 0 (applies to both kinds) |
| `a` — predation | a predator within `eat_radius` of prey kills it |
| `b` — growth from predation | predator gains energy from the kill; above a threshold, reproduces |

`PredatorPreyModel` runs on continuous space — `Point` geometry, an `eat_radius` search — and splits this logic into five explicit phases each tick: movement, metabolism, predation, death, reproduction. Building the starting population requires two agent *kinds* sharing one `GeoDataFrame`:

In [4]:
from dissmodel_abm.models import PredatorPreyModel

rng = np.random.default_rng(0)
n_sheep, n_wolves = 30, 8
kinds = ["sheep"] * n_sheep + ["wolves"] * n_wolves
xs = rng.uniform(bounds[0], bounds[2], n_sheep + n_wolves)
ys = rng.uniform(bounds[1], bounds[3], n_sheep + n_wolves)

pp_gdf = gpd.GeoDataFrame({
    "kind": kinds,
    "energy": [10.0] * (n_sheep + n_wolves),
    "geometry": gpd.points_from_xy(xs, ys),
})

env = Environment(start_time=0, end_time=20)
pp_model = PredatorPreyModel(
    gdf=pp_gdf, bounds=bounds, eat_radius=3.0,
    energy_loss=1.0, energy_gain=5.0,
    reproduce_threshold=15.0, graze_gain=1.5,
)
env.run()
print("Final population:", pp_model.gdf["kind"].value_counts().to_dict())

Running from 0 to 20 (duration: 20)
Final population: {'sheep': 390}


With these particular starting numbers, sheep out-reproduce the wolves' hunting rate entirely and the wolf population collapses to zero — a legitimate outcome of the parameters chosen, not a bug, and exactly the sort of imbalance Exercise 2 asks you to correct by tuning `eat_radius` and `energy_gain` until both populations persist, the way Chapter 20's phase-plane plot showed the continuous version doing.

<div class="admonition warning">
<p class="admonition-title">Watch out</p>
<p>This is a 1:1 <em>architectural</em> port of TerraME's <code>logo/PredatorPrey.lua</code>, not a 1:1 <em>numerical</em> one. Three parameters differ from the original by design: TerraME used per-species reproduction thresholds (rabbits &ge; 30, wolves &ge; 50) where this model has one shared <code>reproduce_threshold</code>; TerraME's predation gain was 20% of the prey's own energy at capture, where this model uses a fixed <code>energy_gain</code>; and TerraME's pasture&rarr;soil&rarr;pasture regrowth cycle has no equivalent here &mdash; <code>graze_gain</code> is a flat, unconditional gain instead. Reproducing the original course's exact numbers means setting these explicitly, not trusting the defaults.</p>
</div>

`SchellingModel`, the third shipped model, applies the same `self.society` discipline to the one-agent-per-cell layout instead — ported directly from TerraME's own `logo` package as a validated reference point, with matching defaults (`dim=25`, 25% free space, `preference=3`):

In [5]:
from dissmodel.geo.vector import vector_grid
from dissmodel_abm.models import SchellingModel

schelling_gdf = vector_grid(dimension=(25, 25), resolution=1)
env = Environment(start_time=0, end_time=30)
schelling = SchellingModel(gdf=schelling_gdf, free_space=0.25, preference=3, seed=0)
env.run()
print("Fraction satisfied:", schelling.fraction_satisfied())

Running from 0 to 30 (duration: 30)
Fraction satisfied: 1.0


A `fraction_satisfied()` of `1.0` means the model converged — every agent ended up with at least `preference` same-type neighbors, Schelling's classic segregation result emerging from nothing more than individually mild, locally-applied preferences.

## What's Not There Yet

Stated directly in the package's own roadmap, not implied by omission: **raster substrate** (a `Society` backed by a NumPy array instead of a `GeoDataFrame`, so model code written against `self.society` would keep working unchanged regardless of substrate — vector support is being hardened first, deliberately); **social networks** (TerraME's message-passing between agents, likely as a thin layer over a graph library keyed by agent ID); and **state machines** (TerraME's `State`/`Jump`/`Flow`, for agents whose behavior depends on a discrete internal mode). Chapter 30's migration guide comes back to this list directly — not every TerraME agent model can be migrated today without a gap, and checking which gap applies before assuming a straightforward port is worth the five minutes it takes.

## Exercises

1. **Why no snapshot?** `Agent` needs no `__init__`-time copy of its data. What makes `agent.energy = 5` immediately visible in `model.gdf`, without an explicit sync step?
2. **Balance the ecosystem.** Starting from the *Case Study* parameters, adjust `eat_radius`, `energy_gain`, and `graze_gain` until both `sheep` and `wolves` survive past tick 20 without either population collapsing to zero. Report the parameters you landed on.
3. **Point agents vs grid agents.** Using the concept-mapping table, explain why `agent.grid_neighbors()` only makes sense for `SchellingModel` and `agent.neighbors(radius)` only for `PredatorPreyModel` — what's structurally different about how each model's agents occupy space?
4. **A gap that matters to you.** Pick one item from *What's Not There Yet* (raster substrate, social networks, state machines) and describe, in a sentence, a model you'd want to build that needs it specifically.

In [ ]:
# Your code here

## Summary

### Key concepts introduced

- `Society`/`Agent` as a protective, substrate-agnostic layer over `self.gdf` — agents read and written as objects, never as raw DataFrame masks, while `Map`, `Chart`, and `ModelExecutor` keep working underneath
- The TerraME-to-`dissmodel-abm` concept mapping, and the point-agent versus one-agent-per-cell distinction it implies
- Three shipped models — `RandomWalkModel`, `PredatorPreyModel`, `SchellingModel` — covering both agent layouts, `SchellingModel` validated directly against TerraME's own `logo` package
- Predator-Prey rebuilt bottom-up from Chapter 20's Lotka-Volterra ODE, with three explicitly documented parameter gaps between the architectural port and the original's exact numbers
- An honest list of what `dissmodel-abm` doesn't do yet — raster substrate, social networks, state machines — that Chapter 30's migration guide checks against directly

Chapter 23 leaves the general-purpose paradigm chapters behind and turns to a specific domain: land use and cover change, built on the same `Model`/`AgentModel` foundation this Part has spent five chapters establishing.

## Further Reading

- Gilbert, N. (2008). *Agent-Based Models*. Sage Publications — a concise case for bottom-up modeling over aggregate equations
- Couclelis, H. (2001). "Why I no longer work with agents." In *Agent-Based Models of Land-Use and Land-Cover Change*
- TerraME's `logo` package documentation, the source of `SchellingModel`'s validated defaults: <https://www.terrame.org/package/logo/models/>
- dissmodel-abm on GitHub: <https://github.com/DisSModel/dissmodel-abm>